In [ ]:
import json
import time
import requests # This library talks to the Cloud
from web3 import Web3

# ==========================================
# 1. CONFIGURATION
# ==========================================
# --- BLOCKCHAIN KEYS ---
INFURA_URL = "https://eth-sepolia.g.alchemy.com/v2/yQnCLGGsGJKH8rISuFzXg"
PRIVATE_KEY = "dc46923fe29ae8b022394667e0b32fe13bc7d12713127465effd9d3eaf745380"

# --- CLOUD KEYS (PINATA) ---
# I have inserted the JWT you just gave me
PINATA_JWT = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ1c2VySW5mb3JtYXRpb24iOnsiaWQiOiJlNTFiZWU2OC0xNjk1LTQxYmUtOTMxMS1mY2M5ZGQ3YTk2ODYiLCJlbWFpbCI6ImFudWp0aGFra2FyMzFAZ21haWwuY29tIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsInBpbl9wb2xpY3kiOnsicmVnaW9ucyI6W3siZGVzaXJlZFJlcGxpY2F0aW9uQ291bnQiOjEsImlkIjoiRlJBMSJ9LHsiZGVzaXJlZFJlcGxpY2F0aW9uQ291bnQiOjEsImlkIjoiTllDMSJ9XSwidmVyc2lvbiI6MX0sIm1mYV9lbmFibGVkIjpmYWxzZSwic3RhdHVzIjoiQUNUSVZFIn0sImF1dGhlbnRpY2F0aW9uVHlwZSI6InNjb3BlZEtleSIsInNjb3BlZEtleUtleSI6IjdkZmY0MTZlMzZjNTlmZGI0ODE1Iiwic2NvcGVkS2V5U2VjcmV0IjoiMDE4MjYwMzhmOTUwM2JjZDA0NzQwMmRiMzI2NDE2MzZiODdhODljM2Q2ZTNlMWEyMTg5MGNmZmJjZDJhOTQzYSIsImV4cCI6MTc5NjkxNjc3M30.RHI7lUpLBWAL_yu2_KNDg_h5ApeDEoPlqMwffsqanYI"

# ==========================================
# 2. SETUP CONNECTIONS
# ==========================================
web3 = Web3(Web3.HTTPProvider(INFURA_URL))

if not web3.is_connected():
    print("❌ Blockchain Connection Failed.")
    exit()

# Address Setup 
SENDER_ADDRESS = web3.to_checksum_address("0xe7987961c7de22ca7242dac44474c0f553912e65")
CONTRACT_ADDRESS = web3.to_checksum_address("0x79DE06d0E587662EEF56eb187c8007Dc253c0e80")
# Dummy colleague address for testing access sharing
COLLEAGUE_ADDRESS = web3.to_checksum_address("0x71C7656EC7ab88b098defB751B7401B5f6d8976F")

# Contract ABI
CONTRACT_ABI = json.loads('''
[
    {"inputs": [{"internalType": "uint256", "name": "_fileId", "type": "uint256"}, {"internalType": "string", "name": "_ipfsCID", "type": "string"}, {"internalType": "string", "name": "_fileName", "type": "string"}, {"internalType": "uint256", "name": "_fileSize", "type": "uint256"}], "name": "uploadFile", "outputs": [], "stateMutability": "nonpayable", "type": "function"},
    {"inputs": [{"internalType": "uint256", "name": "_fileId", "type": "uint256"}, {"internalType": "address", "name": "_user", "type": "address"}], "name": "grantAccess", "outputs": [], "stateMutability": "nonpayable", "type": "function"},
    {"inputs": [{"internalType": "uint256", "name": "_fileId", "type": "uint256"}, {"internalType": "address", "name": "_user", "type": "address"}], "name": "verifyAccess", "outputs": [{"internalType": "bool", "name": "", "type": "bool"}], "stateMutability": "view", "type": "function"}
]
''')

contract = web3.eth.contract(address=CONTRACT_ADDRESS, abi=CONTRACT_ABI)

# ==========================================
# 3. FUNCTION: UPLOAD TO CLOUD (PINATA)
# ==========================================
def upload_to_ipfs(filename):
    """
    Sends the file to Pinata (IPFS Cloud) and gets a Hash back.
    """
    url = "https://api.pinata.cloud/pinning/pinFileToIPFS"
    
    headers = {
        "Authorization": f"Bearer {PINATA_JWT}"
    }

    print(f"\n☁️  Uploading '{filename}' to IPFS Cloud...")
    try:
        with open(filename, 'rb') as file_data:
            files = {'file': file_data}
            response = requests.post(url, headers=headers, files=files)
            
        if response.status_code == 200:
            data = response.json()
            ipfs_hash = data['IpfsHash']
            print(f"✅ Cloud Upload Success! Real CID: {ipfs_hash}")
            return ipfs_hash
        else:
            print(f"❌ Cloud Error: {response.text}")
            exit()
            
    except FileNotFoundError:
        print(f"❌ Error: File '{filename}' not found.")
        exit()

# ==========================================
# 4. FUNCTION: UPLOAD TO BLOCKCHAIN
# ==========================================
def send_transaction(function_call, description):
    print(f"\n⏳ {description}...")
    nonce = web3.eth.get_transaction_count(SENDER_ADDRESS)
    
    tx_data = function_call.build_transaction({
        'chainId': 11155111,
        'gas': 400000,   # Optimized gas limit
        'maxFeePerGas': web3.to_wei('50', 'gwei'),
        'maxPriorityFeePerGas': web3.to_wei('2', 'gwei'),
        'nonce': nonce,
        'from': SENDER_ADDRESS
    })
    
    signed_tx = web3.eth.account.sign_transaction(tx_data, PRIVATE_KEY)
    tx_hash = web3.eth.send_raw_transaction(signed_tx.raw_transaction)
    
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
    print(f"✅ Confirmed on Blockchain! Gas Used: {receipt.gasUsed}")
    print(f"🔗 Tx: https://sepolia.etherscan.io/tx/{receipt.transactionHash.hex()}")
    return receipt

# ==========================================
# 5. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    
    # A. Generate unique ID based on current time (prevents collisions)
    file_id = int(time.time())
    
    # B. Create a real file to test with
    filename = "Final_Dissertation_Test.txt"
    with open(filename, "w") as f:
        f.write(f"Confidential Project Data. Generated at {file_id}")
    
    print(f"--- 🚀 STARTING SYSTEM TEST (File ID: {file_id}) ---")

    # STEP 1: UPLOAD TO CLOUD (Get the Real CID)
    real_ipfs_hash = upload_to_ipfs(filename)
    
    # STEP 2: LINK TO BLOCKCHAIN
    # We send the Real CID (from step 1) to the Smart Contract
    upload_call = contract.functions.uploadFile(file_id, real_ipfs_hash, filename, 1024)
    send_transaction(upload_call, f"Linking Cloud File (CID: {real_ipfs_hash[:10]}...) to Blockchain")

    # STEP 3: VERIFY ACCESS
    print("\n🔍 Verifying Owner Access...")
    has_access = contract.functions.verifyAccess(file_id, SENDER_ADDRESS).call()
    print(f"👉 Access Status: {'GRANTED' if has_access else 'DENIED'}")
    
    print("\n--- ✅ TEST COMPLETE. TAKE SCREENSHOT NOW! ---")

--- 🚀 STARTING SYSTEM TEST (File ID: 1765498724) ---

☁️  Uploading 'Final_Dissertation_Test.txt' to IPFS Cloud...
✅ Cloud Upload Success! Real CID: QmWyJwBBtdLmro3Rm9QHJjykh9M7SViZdB5JdmoKVxEChh

⏳ Linking Cloud File (CID: QmWyJwBBtd...) to Blockchain...
✅ Confirmed on Blockchain! Gas Used: 206409
🔗 Tx: https://sepolia.etherscan.io/tx/dd803941d24745895ce9ed7cd2159f392f2f01bfd6afa7acdd1bc4d1e7fa099d

🔍 Verifying Owner Access...
👉 Access Status: GRANTED

--- ✅ TEST COMPLETE. TAKE SCREENSHOT NOW! ---
